# 1D linearized shallow-water equation with wind stress

The one-dimensional shallow-water equations can be simplified to a 1D wave-equation with wind forcing:

$\partial h/\partial t + \partial Hu / \partial x = 0$

$\partial u/\partial t + g \partial h / \partial x = \tau/(\rho H) -g u|u|/(H C^2)$

where $h$ denotes the water-height above the reference, $u$ the velocity, $D$ the depth below the reference, $H=D+h$ the total water depth and $t,x$ time and space. In the quadratic friction term $C$ denotes the Chezy friction coefficient.

At the left boundary $q_{left}$ denotes the inflow and $q_{right}$ the outflow at the right boundary, i.e. the sign is aligned with $u$.

$u(x=0,t)=q_{left}(t)/(H W)$

$u(x=L,t)=q_{right}(t)/(H W)$

The equations and approach are similar to the model_1d_wave_periodic.jl 
model. Here the equations are not fully linear, on a 'closed' domain,
and with a wind-stress forcing term.


In [ ]:
# goto folder of this notebook (If you move this notebook then also copy the Project.toml file and run instantiate to install the packages)
cd(@__DIR__)
# load the environment
using Pkg 
Pkg.activate(".")  # activate the environment for this notebook
# Pkg.instantiate()
# load the packages
using OrdinaryDiffEq
using ComponentArrays
using Plots
using JLD2
using Random


In [ ]:
# Load model definition
include("model_1d_surge_wave.jl")

In [ ]:
function initial_state_rest(f::Wave1DSurge_cpu)
    x_h = f.dx/2:f.dx:(f.L-f.dx/2)
    x_u = 0.0:f.dx:f.L
    h = zeros(length(x_h))
    u = zeros(length(x_u))
    return ComponentVector(h=h, u=u)
end


function initial_state_bump(f::Wave1DSurge_cpu, h0=1.0, w=0.05, c=0.5,u0=0.0)
    x_center=f.L*c
    width=f.L*w
    x_h = f.dx/2:f.dx:(f.L-f.dx/2)
    x_u = 0.0:f.dx:f.L
    h = h0.*exp.(-((x_h .- x_center).^2) ./ (2 * width^2))
    u = u0.*exp.(-((x_u .- x_center).^2) ./ (2 * width^2))
    x = ComponentVector(h=h,u=u)
    return x
end

function initial_state_tilt(f::Wave1DSurge_cpu; h_start=-1.0, h_end=1.0)
    x_h = f.dx/2:f.dx:(f.L-f.dx/2)
    x_u = 0.0:f.dx:f.L
    h = h_start .+ (h_end - h_start) .* (x_h ./ f.L)  
    u = zeros(length(x_u))
    return ComponentVector(h=h, u=u)
end

In [ ]:
function make_target_wind(rng)
    hour = 3600.0
    minute = 60.0

    background_amplitude = 0.6 + 0.4 * rand(rng)       # 0.6–1.0 Pa
    background_period = (2.5 + 2.0 * rand(rng)) * hour
    background_phase = 2π * rand(rng)

    storm_amplitude = 1.4 + 0.8 * rand(rng)             # 1.4–2.2 Pa
    storm_center = (2.8 + 1.0 * rand(rng)) * hour
    rise_width = (0.45 + 0.30 * rand(rng)) * hour
    decay_width = (1.0 + 0.50 * rand(rng)) * hour

    gust_period = (25.0 + 40.0 * rand(rng)) * minute
    gust_phase = 2π * rand(rng)

    second_amplitude = storm_amplitude * (0.15 + 0.20 * rand(rng))
    second_center = storm_center + (1.2 + 0.6 * rand(rng)) * hour
    second_width = (0.25 + 0.25 * rand(rng)) * hour

    function tau(t)
        width = t <= storm_center ? rise_width : decay_width

        primary_envelope =
            exp(-0.5 * ((t - storm_center) / width)^2)

        secondary_envelope =
            exp(-0.5 * ((t - second_center) / second_width)^2)

        background =
            background_amplitude *
            sin(2π * t / background_period + background_phase)

        primary_storm =
            storm_amplitude *
            primary_envelope *
            (0.75 + 0.25 * sin(2π * t / gust_period + gust_phase))

        secondary_storm =
            second_amplitude * secondary_envelope

        return background + primary_storm + secondary_storm
    end

    return tau
end

In [ ]:
# setting up the problem
# parameters
nx=100     # number of spatial points
L=100.0e3  # length of the domain
dx=L/nx    # spatial step
rho=1000.0 # density of water
g = 9.81   # acceleration due to gravity
grid_u=collect(0.0:dx:L) # grid for velocity
grid_h=collect(dx/2:dx:(L-dx/2)) # grid for water levels
D_min=20.0 # minimum depth of the water
D_max=20.0 # maximum depth of the water
D_edges=[0.3,0.7] .* L # edges of the depth change
D_widths=[0.08, 0.08] .* L # widths of the depth change
D=depth_profile(grid_u,D_min,D_max,D_edges,D_widths)
W=100.0 # width
C = 50.0    # Chezy friction factor
n_ens = 10 # number of ensemble members



storm_center = 3.2*3600.0
storm_width  = 1.1*3600.0
storm_envelope(t) = exp(-((t-storm_center)^2)/(2*storm_width^2))

# wind stress: background oscillation + storm burst + gusts
tau_base_amp = 1.0
tau_storm_amp = 2.0
T_wind_bg = 3*3600.0
T_wind_gust = 45*60.0
tau(t) = tau_base_amp*sin(2*π*t/T_wind_bg) +
         tau_storm_amp*storm_envelope(t)*(0.7 + 0.3*sin(2*π*t/T_wind_gust + π/3)) 

function constant_func(t,value=0.0)
    return value
end
# q_left(t)=constant_func(t,0.0)
# q_right(t)=constant_func(t,0.0) #Note q_left-q_right is the nett influx

# f=Wave1DSurge_cpu(g,D,L,W,dx,nx,rho,C,tau,q_left,q_right)
# x0_all = []
# for i in 1:n_ens
#     ens_start = -1 + rand() 
#     ens_end = 0 + rand()
#     push!(x0_all, initial_state_rest(f))
# end


# time span
t_start=0.0
t_end= 12*3600.0

# spatial grid 
x_h = grid_h
x_u = grid_u
x=ComponentVector(h=x_h,u=x_u)

In [ ]:
# # test steps of the pde model
# h_avg=similar(x0.u)
# avg_h!(h_avg,x0.h,dx)
# #@show h_avg[1:10:end]'
# H=h_avg .+ D
# @show H[1:10:end]'
# Hu=H .* x0.u
# @show x0.u[1:10:end]'
# @show Hu[1:10:end]'

In [ ]:
# # plot the initial state
# p1=plot(1e-3*x_h, x0.h, label="h(t=0)", xlabel="x", ylabel="h")
# p2=plot(1e-3*x_u, x0.u, label="u(t=0)", xlabel="x", ylabel="u")
# p3=plot(1e-3*x_u, -D, label="depth D", xlabel="x", ylabel="D")
# plot(p1, p2, p3, layout=(3,1), size=(1000,600),title="Initial state of the 1D wave equation")

In [ ]:
rng = MersenneTwister(2026)

q_left(t) = 0.0
q_right(t) = 0.0

tau_all = Function[]
f_all = []
sol_all = []

for i in 1:n_ens
    tau_i = make_target_wind(rng)

    f_i = Wave1DSurge_cpu(
        g, D, L, W, dx, nx, rho, C,
        tau_i, q_left, q_right,
    )

    x0_i = initial_state_rest(f_i)
    prob_i = ODEProblem(f_i, x0_i, (t_start, t_end))

    println("Running target simulation $i of $n_ens")
    sol_i = solve(prob_i, Rodas5())

    push!(tau_all, tau_i)
    push!(f_all, f_i)
    push!(sol_all, sol_i)
end
nothing

In [ ]:
# visualize tidal boundary forcing and wind stress
times_plot = collect(t_start:300.0:t_end)  # every 5 minutes
hours_plot = times_plot ./ 3600.0

qL_plot = [q_left(t) for t in times_plot]
qR_plot = [q_right(t) for t in times_plot]
tau_plot = [tau_all[3](t) for t in times_plot]

p_q = plot(hours_plot, qL_plot, label="q_left", xlabel="time [h]", ylabel="transport [m^3/s]", lw=2)
plot!(p_q, hours_plot, qR_plot, label="q_right", lw=2, ls=:dash)

p_tau = plot(hours_plot, tau_plot, label="tau", xlabel="time [h]", ylabel="wind stress [N/m^2]", lw=2, color=:black)

plot(p_q, p_tau, layout=(2,1), size=(1000,600), title="")

In [ ]:
# create a movie of the solution

@time begin
    # create a movie of the solution
    anim = @animate for i in 1:length(sol_all[3].t)
        p1 = plot(1e-3*x_h, sol_all[3][i].h, label="h(t=$(round(sol_all[3].t[i]/3600,digits=1)))", xlabel="x", ylabel="h", ylim=(-3, 3))
        plot!(p1,1e-3*x_u,-0.1*f.D,label=false) #depth with false scaling
        p2 = plot(1e-3*x_u, sol_all[3][i].u, label="u(t=$(round(sol_all[3].t[i]/3600,digits=1)))", xlabel="x", ylabel="u", ylim=(-3, 3))
        plot(p1, p2, layout=(2,1), size=(800,600), title="Solution of the 1D wave equation with winds")
    end
    gif(anim, "wave1d_surge.gif", fps=30)
end


In [ ]:
# save the solution to a JLD2 file
cfl_out=0.7 # CFL number for output
dt_out_auto=cfl_out*f.dx/(sqrt(maximum(f.D)*f.g)) # time step for output, based on CFL condition
println("time-step with CFL=0.7 is $(dt_out_auto)")
dt_out=60.0 #manual value
times=collect(0.0:dt_out_auto:t_end) # output times
solution=[]
for i in 1:n_ens
    sol = sol_all[i]
    temp_sol = []
    for t in times
        h_value=sol(t).h
        u_value=sol(t).u
        tau_value= tau_all[i](t)
        q_left_value=f.q_left(t)
        q_right_value=f.q_right(t)
        x=ComponentVector(h=h_value,u=u_value,tau=tau_value,q_left=q_left_value,
            q_right=q_right_value)
        push!(temp_sol, x)
    end
    push!(solution, temp_sol)
end
@show length(solution)
# save the solution to a JLD2 file
save("wave1d_tide_constant_10_forcings_pretrain.jld2", 
    "solution", solution,
    "times", times,
    "params", (g=f.g, C=f.C, rho=f.rho, W=f.W, D=f.D, L=f.L, 
        dx=f.dx, nx=f.nx),
    "grid", ComponentVector(h=x_h,u=x_u),
    "cfl_out", cfl_out,
    "dt_out", dt_out_auto,
    "description", "Solution of the 1D shallow water equation with wind stress.")


In [ ]:
# # load the solution from the JLD2 file for testing
# ff=load("wave1d_tide_ens_test.jld2")

# solution = ff["solution"]
# n_ens = length(solution)
# n_steps = length(solution[1]) - 1
# n_h_points = length(solution[1][1].h)
# n_u_points = length(solution[1][1].u)
# @show n_ens, n_steps, n_h_points, n_u_points

